<a href="https://colab.research.google.com/github/inthecrowsnest/DeepLearningProject/blob/main/data_processing_m1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Date : 14th October, 2026

We are using subtitle data for our project.

In [20]:
!pip install huspacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.8/92.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigquery-magics 0.14.0 requires packaging>=24.2.0, but you have packaging 21.3 which is incompatible.
google-cloud-bigquery 3.45.2 requires packaging>=24.2.0, but you have packaging 21.3 which is incompatible.
astropy 7.2.2 requires packaging>=22.0.0, but you have packaging 21.3 which is incompatible.
wheel 0.48.0 requires packaging>=24.0, but you have packaging 21.3 which is incompatible.
langchain-core 1.6.6 requires packaging>=23.2.0, but you have packaging 21.3 which is incompatible.
jupyter-server 

In [3]:
import pandas as pd
import huspacy
import spacy


# Download the latest CPU optimized model
huspacy.download()

['/usr/bin/python3', '-m', 'pip', 'install', 'hu_core_news_lg @ https://huggingface.co/huspacy/hu_core_news_lg/resolve/v3.8.0/hu_core_news_lg-any-py3-none-any.whl']


In [11]:
seg = pd.read_csv('1956727551.csv')
labels = pd.read_csv('words.csv')

# Selecting the actual subtitle text column '__text' from seg
# and lemma & label columns from labels
seg_selected = seg[['__text']].rename(columns={'__text': 'text'})



In [12]:
# Pre-processing of the column
# Load the model using spacy.load(...)
nlp = spacy.load("hu_core_news_lg")

tokens_list, lemmas_list = [], []
for doc in nlp.pipe(texts, batch_size=50, disable=["ner", "parser"]):
    words = [t for t in doc
             if not t.is_stop and not t.is_punct and not t.is_space and t.is_alpha]
    tokens_list.append([t.text.lower() for t in words])
    lemmas_list.append([t.lemma_.lower() for t in words])

seg_selected['tokens'] = tokens_list   # stop words removed
seg_selected['lemmas'] = lemmas_list   # stop words removed

# One row per word (token + its lemma side by side)
tokens_df = (seg_selected
             .explode(['tokens', 'lemmas'])
             .dropna(subset=['tokens'])
             .rename(columns={'tokens': 'token', 'lemmas': 'lemma'})
             [['text', 'token', 'lemma']]
             .reset_index(names='line_id'))

tokens_df = tokens_df[['token', 'lemma']]

tokens_df.to_csv('tokens_dataset.csv', index=False, encoding='utf-8-sig')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>